# AI-Based Flood Prediction & Resilience Planning
### Exploratory Data Analysis, Model Inference & Resilience Demonstration
**4th-Year B.Tech Major Project**

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.geospatial.risk_mapper import SpatialRiskEngine
from src.explainability.shap_explainer import FloodShapExplainer
from src.resilience.recommendation_engine import ResilienceEngine

print("Project root configured:", PROJECT_ROOT)

## 1. Load Processed Dataset & Inspect Class Distribution

In [ ]:
df_full = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "processed", "full_features.csv"))
print(f"Total dataset shape: {df_full.shape}")
print("\nClass Distribution (flood_occurred):")
print(df_full['flood_occurred'].value_counts(normalize=True).round(4) * 100)
print("\nHazard Level Counts:")
print(df_full['hazard_level'].value_counts().sort_index())

## 2. Load Model Comparison Benchmarks

In [ ]:
with open(os.path.join(PROJECT_ROOT, "outputs", "metrics", "model_comparison.json")) as f:
    metrics = json.load(f)

df_metrics = pd.DataFrame(metrics).T[['accuracy', 'precision', 'recall', 'f1', 'pr_auc', 'roc_auc', 'brier_score']]
df_metrics.sort_values('f1', ascending=False)

## 3. Real-Time Inference & Risk Scoring Demo

In [ ]:
risk_engine = SpatialRiskEngine()

# Sample Case: Aluva Station during heavy monsoon saturation
aluva_case = {
    'rainfall_1d': 92.5,
    'rainfall_3d': 182.0,
    'rainfall_7d': 285.0,
    'rainfall_14d': 405.0,
    'rainfall_anomaly_30d': 85.0,
    'api_14d': 140.0,
    'temp_mean_3d': 24.8,
    'humidity_mean_3d': 93.5,
    'soil_moisture_root_zone': 0.44,
    'soil_moisture_deep': 0.41,
    'soil_saturation_ratio': 0.92,
    'elevation_m': 10.0,
    'slope_deg': 4.72,
    'dist_river_km': 0.3,
    'twi_proxy': 7.15,
    'elevation_risk_factor': 0.82,
    'sar_vv': -15.8,
    'sar_vh': -21.4,
    'sar_cross_ratio': 1.35,
    'ndwi_proxy': 0.62,
    'ndvi_proxy': 0.48,
    'population_density': 1359.0,
    'exposure_index': 0.85,
    'drainage_vulnerability': 0.78,
    'vulnerability_index': 0.81
}

result = risk_engine.compute_point_risk(aluva_case)
print("Risk Computation Output:")
for k, v in result.items():
    print(f"  {k}: {v}")

## 4. TreeSHAP Explanation Breakdown

In [ ]:
shap_explainer = FloodShapExplainer()
explanation = shap_explainer.explain_instance(aluva_case)

print(f"Calibrated Flood Probability: {explanation['probability'] * 100:.2f}%")
print("\nTop Factors Increasing Risk:")
for f in explanation['factors_increasing_risk']:
    print(f"  [+] {f['readable_name']}: Value={f['value']}, SHAP Contribution=+{f['shap_value']}")

print("\nTop Factors Mitigating Risk:")
for f in explanation['factors_decreasing_risk']:
    print(f"  [-] {f['readable_name']}: Value={f['value']}, SHAP Contribution={f['shap_value']}")

## 5. Resilience Planning Directives

In [ ]:
res_engine = ResilienceEngine()
recs = res_engine.generate_recommendations(
    hazard_prob=result['hazard_probability'],
    exposure_index=result['exposure_index'],
    vulnerability_index=result['vulnerability_index'],
    elevation_m=10.0,
    dist_river_km=0.3,
    rainfall_3d=182.0,
    basin_name="Periyar",
    location_name="Aluva"
)

print("=" * 70)
print(f"LOCATION: {recs['location']} ({recs['basin']} Basin)")
print(f"URGENCY: {recs['urgency_level']}")
print(f"DISCLAIMER: {recs['disclaimer']}")
print("=" * 70)
print("\nImmediate Emergency Directives:")
for a in recs['emergency_actions_immediate']:
    print(f"  * [{a['target_agency']}] {a['action']}: {a['detail']}")